# Iterate Production Buildings

This notebook shows how to:
1. Find production buildings
2. Extract their properties
3. Get production chains (inputs/outputs)
4. Use `format_production_list()` for construction materials

In [1]:
from assetextractor.extraction.utils import Config
from assetextractor.parsing.core.assets import AssetCache
from assetextractor.parsing.core.texts import StandardTextConverter

# Load assets
config = Config.from_json("config.json")
assets = AssetCache.load(config)
texts = assets.texts

# Set language
LANGUAGE = "english"
texts.converter = StandardTextConverter(LANGUAGE)

print("Assets loaded!")

Assets loaded!


## Find Production Building Templates

In [2]:
# Find templates with "Production" or "Factory" or "Farm"

for template in assets.templates.groups["Objects"]["Buildings"]["Factories"]: # use the group under which all production building templates are listed
    print(f"  {template.name}: {len(template.assets)} assets")

  Multifactory: 0 assets
  RecipeFarm: 0 assets
  Monument: 7 assets
  Production: 84 assets
  Production Area: 9 assets
  Production Field: 21 assets
  Production Marsh: 5 assets
  Production Marsh Area: 3 assets
  Production Marsh Pasture: 2 assets
  SlotFactoryBuilding7: 16 assets
  ProductionModuleSilo: 2 assets


## Iterate Production Buildings

Let's iterate through `Production` and extract basic information:

In [ ]:
# Check if template exists
if "Production" in assets.templates.elements:
    factory_template = assets.templates["Production"]
    
    print(f"Found {len(factory_template.assets)} factory buildings\n")
    print("First 5 factory buildings:")
    
    for i, building in enumerate(list(factory_template.assets)[:5]):
        name = building.text() if building.text else None
        guid = building.guid
        
        print(f"\n{i+1}. {name} (GUID: {guid})")
        
        # Get basic properties
        if hasattr(building, 'Cost'):
            costs = building.find("Cost.Costs")
            if costs:
                print(f"   Construction costs: {len(costs)} items")
        
        if hasattr(building, 'Maintenance'):
            maintenance = building.find("Maintenance.Maintenances")
            if maintenance:
                print(f"   Maintenance: {len(maintenance)} items")
else:
    print("Template 'Production' not found")

Found 84 factory buildings

First 5 factory buildings:

1. Fishing Hut (GUID: 2955)
   Construction costs: 6 items
   Maintenance: 2 items

2. Scomber's Shack (GUID: 2956)
   Construction costs: 6 items
   Maintenance: 2 items

3. Salt Ponds (GUID: 2957)
   Construction costs: 6 items
   Maintenance: 2 items

4. Snailery (GUID: 8580)
   Construction costs: 6 items
   Maintenance: 2 items

5. Sand Refinery (GUID: 2958)
   Construction costs: 6 items
   Maintenance: 2 items


## Extract Production Chain

Get inputs and outputs for a production building:

In [4]:
from assetextractor.parsing.core.assets import Asset


def extract_production_info(building: Asset):
    """Extract input and output products from a building."""
    inputs = []
    outputs = []
    
    # Try to find FactoryBase with production info
    factory_base = building.find("FactoryBase")
    if factory_base:
        # Get inputs
        input_list = factory_base.find("FactoryInputs")
        if input_list:
            inputs = assets.properties.ui_text_cache.format_product_list(input_list)
        
        # Get outputs
        output_list = factory_base.find("FactoryOutputs")
        if output_list:
            outputs = assets.properties.ui_text_cache.format_product_list(output_list)
    
    return inputs, outputs

# Test with first factory
if "Production" in assets.templates.elements:
    factory = assets.get(2994)
    name = factory.text() if factory.text else 'N/A'
    
    inputs, outputs = extract_production_info(factory)
    
    print(f"Production chain for: {name}\n")
    
    if inputs:
        print("Inputs:")
        for ui in inputs:
            print(f"  - {ui.value}t {ui.text()}")
    
    if outputs:
        print("\nOutputs:")
        for ui in outputs:
            print(f"  - {ui.value}t {ui.text()}")

Production chain for: Tannery

Inputs:
  - +1t Pigs
  - +1t Salt

Outputs:
  - +1t Leather


## Next Steps

- `04_buffs_and_effects.ipynb` - Extract buffs that affect these buildings
- `05_construction_materials.ipynb` - Detailed construction cost analysis
- `06_backtrack_effects.ipynb` - Find items that improve production